In [1]:
%matplotlib inline
%config InlineBackend.figure_formats = ['svg']

from IPython.display import clear_output

import json
import yaml

import copy
import os
import subprocess
import sys
import time

import tempfile
import h5py

import pandas as pd
import numpy as np
import matplotlib as mpl

import matplotlib.pyplot as plt

import pandas as pd
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
import h5py

plt.rcParams['axes.prop_cycle'] = plt.cycler(color=['olivedrab', 'steelblue', 'firebrick', 'goldenrod'])
plt.rcParams['axes.formatter.use_mathtext'] = True
plt.rcParams['axes.formatter.useoffset'] = False
plt.rcParams['axes.formatter.limits'] = (0, 0)
plt.rcParams['figure.figsize'] = [6,4]
plt.rcParams['figure.constrained_layout.use'] = True
plt.rcParams['legend.frameon'] = False
plt.rcParams['xtick.minor.visible'] = True
plt.rcParams['ytick.minor.visible'] = True

template_file = 'PSLS/examples/psls.yaml'

with open(template_file, 'r') as file:
    config_dict = yaml.safe_load(file)
    print(json.dumps(config_dict, indent=4, sort_keys=False))

{
    "Observation": {
        "QuarterDuration": [
            90.0,
            90.0,
            90.0
        ],
        "MasterSeed": 1704040900,
        "Gaps": {
            "Enable": 1,
            "Seed": -1,
            "InterQuarterGapDuration": 3.0,
            "RandomGapDuration": 0.0,
            "RandomGapTimeFraction": 0.5,
            "RandomGapStep": 0.0,
            "PeriodicGapCadence": 5.0,
            "PeriodicGapDuration": 20.0,
            "PeriodicGapJitter": 2.0,
            "PeriodicGapStep": 0.0
        }
    },
    "Instrument": {
        "Sampling": 25.0,
        "IntegrationTime": 21.0,
        "GroupID": [
            1,
            2,
            3,
            4
        ],
        "NCamera": 6,
        "TimeShift": 6.25,
        "RandomNoise": {
            "Enable": 1,
            "Type": "PLATO_SIMU",
            "NSR": 73.0
        },
        "Systematics": {
            "Enable": 1,
            "Table": "systematics/PLATO_systematics_BOL_V2.npy",
  

In [2]:
psls_template = {
    'Observation': {
        'QuarterDuration': [90.0, 90.0, 90.0],
        'MasterSeed': 1704040900,
        'Gaps': {
            'Enable': 1,
            'Seed': -1,
            'InterQuarterGapDuration': 3.0,
            'RandomGapDuration': 0.0,
            'RandomGapTimeFraction': 0.5,
            'RandomGapStep': 0.0,
            'PeriodicGapCadence': 5.0,
            'PeriodicGapDuration': 20.0,
            'PeriodicGapJitter': 2.0,
            'PeriodicGapStep': 0.0
        }
    },
    'Instrument': {
        'Sampling': 25.0,
        'IntegrationTime': 21.0,
        'GroupID': [1, 2, 3, 4],
        'NCamera': 6,
        'TimeShift': 6.25,
        'RandomNoise': {
            'Enable': 1,
            'Type': 'PLATO_SIMU',
            'NSR': 73.0
        },
        'Systematics': {
            'Enable': 1,
            'Table': 'systematics/PLATO_systematics_BOL_V2.npy',
            'Version': 2,
            'DriftLevel': 'any',
            'Seed': -1
        }
    },
    'Star': {
        'Mag': 10.0,
        'ID': 12069449,
        'ModelType': 'grid',
        'ModelDir': 'models/',
        'ModelName': 'grid.hdf5',
        'ES': 'ms',
        'Teff': 5750.0,
        'Logg': 4.353,
        'SurfaceRotationPeriod': 0.0,
        'CoreRotationFreq': 0.0,
        'Inclination': 0.0
    },
    'Oscillations': {
        'Enable': 1,
        'numax': 179.3,
        'delta_nu': 13.68,
        'DPI': 80.58,
        'q': 0.15,
        'SurfaceEffects': 1,
        'Seed': -1
    },
    'Activity': {
        'Enable': 1,
        'Sigma': 40.0,
        'Tau': 0.2,
        'Seed': -1,
        'Spot': {
            'Enable': 0,
            'dOmega': 0.0,
            'MuStar': 0.59,
            'MuSpot': 0.78,
            'Radius': [2.5, 2.5, 2.5],
            'Latitude': [0.0, 20.0, 40.0],
            'Longitude': [0.0, 0.0, 0.0],
            'Lifetime': [10, 30, 50],
            'TimeMax': [-1, -1, -1],
            'Contrast': [0.7, 0.8, 0.6],
            'Modulation': 0.0,
            'Seed': -1
        },
        'Flare': {
            'Enable': 0,
            'MeanPeriod': 2,
            'Amplitude': 2500.0,
            'UpDown': 0.1,
            'MeanDuration': -1,
            'DurationDispersion': -1,
            'Seed': -1
        }
    },
    'Granulation': {
        'Enable': 1,
        'Type': 1,
        'Seed': -1
    },
    'Transit': {
        'Enable': 1,
        'PlanetRadius': 0.5,
        'OrbitalPeriod': 10.0,
        'PlanetSemiMajorAxis': 1.0,
        'OrbitalAngle': 0.0,
        'LimbDarkeningCoefficients': [0.25, 0.75]
    },
    'External': {
        'Enable': 0,
        'FilePath': 'examples/external_example.txt'
    }
}

In [3]:
def simulate(total_sims = 5):

    start_batch = time.time()

    id_str = time.strftime('%Y%m%d%H%M%S', time.gmtime(start_batch))
    attr_str = time.strftime('%d.%m.%Y, %H:%M:%S', time.gmtime(start_batch))

    file_name = f'PSLS/data/{id_str}.hdf5'

    with h5py.File(file_name, 'x') as f:
        f.attrs['description'] = f'Batch Simulation Results ({total_sims} Runs)'
        f.attrs['date_created'] = attr_str
        f.attrs['user'] = 'Fritz Ali Agildere'

    initial_status = f'│ progress |{'░' * 50}| 0% [running 1/{total_sims}] (00:00:00/--:--:--)          '
    
    print(f'{initial_status}')
    
    i = 0
    while i < total_sims:
    
        config = copy.deepcopy(psls_template)
    
        config['Star']['ID'] = i
        
        if i != 0:
            perc = i / total_sims * 100
            bar = '█' * int(perc // 2) + '░' * (50 - int(perc // 2))
            status = f'│ progress |{bar}| {perc:.0f}% [running {i+1}/{total_sims}] ({elapsed_str}/{estimated_str})          '
    
            clear_output(wait=True)
            print(f'{status}')
    
        with tempfile.NamedTemporaryFile(suffix='.yaml', dir='PSLS', mode='w', delete=True) as tf:
            yaml.dump(config, tf, default_flow_style=False, sort_keys=False)
            tf.flush()
    
            sys.stdout.write(f'└── initializing simulation {i + 1}          ')
            sys.stdout.flush()
            time.sleep(0.1)
            sys.stdout.write(f'\r└─┬ initializing simulation {i + 1}          \n')
            sys.stdout.write(f'  └── sampling parameters [active]          ')
            sys.stdout.flush()
            
            time.sleep(1)
            
            sys.stdout.write(f'\r  ├── sampling parameters [done]          \n')
            sys.stdout.write(f'  └── generating lightcurve [active]          ')
            sys.stdout.flush()
            
            %cd -q PSLS
            !echo '' | ./psls.py -o data {tf.name}
            %cd -q ..
    
            sys.stdout.write(f'\r  ├── generating lightcurve [done]          \n')
            sys.stdout.write(f'  └── saving dataframe [active]          ')
            sys.stdout.flush()

            group_name = f'run_{i + 1:05d}'
            with h5py.File(file_name, 'a') as f:
                run = f.create_group(group_name)
                with open(tf.name, 'r') as cfg:
                    run.attrs['config_used'] = cfg.read()
                with open(f'PSLS/data/{i + 1:010d}.txt', 'r') as txt:
                    content = txt.read()
                    dt = h5py.string_dtype(encoding='utf-8')
                    subgroup.create_dataset('txt', data=content, dtype=dt)
                dat = f[group_name].create_group('dat')
                time_s, flux_var_ppm, flag = np.genfromtxt(f'PSLS/data/{i + 1:010d}.dat', unpack=True, skip_header=5)
                dat.create_dataset('time_s', data=time_s, compression='gzip')
                dat.create_dataset('flux_var_ppm', data=flux_var_ppm, compression='gzip')
                dat.create_dataset('flag', data=flag, compression='gzip')
                modes = f[group_name].create_group('modes')
                n, l, m, nu, gamma, h, I_Imax, dnu, split = np.genfromtxt(f'PSLS/data/{i + 1:010d}.modes', unpack=True, skip_header=1)
                dat.create_dataset('n', data=n, compression='gzip')
                dat.create_dataset('l', data=l, compression='gzip')
                dat.create_dataset('m', data=m, compression='gzip')
                dat.create_dataset('nu', data=nu, compression='gzip')
                dat.create_dataset('gamma', data=gamma, compression='gzip')
                dat.create_dataset('h', data=h, compression='gzip')
                dat.create_dataset('I_Imax', data=I_Imax, compression='gzip')
                dat.create_dataset('dnu', data=dnu, compression='gzip')
                dat.create_dataset('split', data=split, compression='gzip')

        current_run = time.time()
        
        elapsed_batch = current_run - start_batch
        
        average_time = elapsed_batch / (i + 1)
        
        estimated_batch = elapsed_batch + (total_sims - (i + 1)) * average_time
    
        elapsed_str = time.strftime('%H:%M:%S', time.gmtime(elapsed_batch))
        estimated_str = time.strftime('%H:%M:%S', time.gmtime(estimated_batch))
    
        i += 1

    final_status = f'│ progress |{'█' * 50}| 100% [finished {i}/{total_sims}] ({elapsed_str}/{estimated_str})          '

    clear_output(wait=True)
    print(f'{final_status}')

In [4]:
simulate(2)

│ progress |░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░| 0% [running 1/2] (00:00:00/--:--:--)          
└─┬ initializing simulation 1          
  ├── sampling parameters [done]            
  ├── generating lightcurve [done]            
  └── saving dataframe [active]          

NameError: name 'subgroup' is not defined

In [ ]:
with h5py.File('PSLS/data/20260430093146.hdf5', 'r') as f:
    print(f.keys())

In [ ]:
%%time

time, flux_var, flag = np.genfromtxt("PSLS/data/0000000000.dat", unpack=True, skip_header=5)
time, flux_var, flag

n, l, m, nu, gamma, h, I_Imax, dnu, split = np.genfromtxt("PSLS/data/0000000000.modes", unpack=True, skip_header=1)
n, l, m, nu, gamma, h, I_Imax, dnu, split